In [2]:
import os
import time
from io import BytesIO
from pathlib import Path

import requests
from PIL import Image

# ModelScope 异步图像生成配置
BASE_URL = "https://api-inference.modelscope.cn/"
DEFAULT_MODEL = "Tongyi-MAI/Z-Image-Turbo"
POLL_INTERVAL = 5          # 任务状态轮询间隔(秒)
REQUEST_TIMEOUT = 30       # 单次 HTTP 请求超时(秒)


def generate_image(
    prompt: str,
    save_path: str | Path = "result_image.jpg",
    model: str = DEFAULT_MODEL,
    poll_interval: int = POLL_INTERVAL,
) -> Path:
    """调用 ModelScope 异步图像生成 API,轮询至完成后保存到本地。
    
    提交流程: 提交异步任务 → 轮询状态 → SUCCEED 后下载首张图。
    
    Args:
        prompt: 图片描述文本。
        save_path: 图片保存路径,默认 "result_image.jpg"。
        model: ModelScope 模型 ID,默认 Tongyi-MAI/Z-Image-Turbo。
        poll_interval: 任务状态轮询间隔(秒),默认 5。
    
    Returns:
        保存后的图片绝对路径(Path)。

    Raises:
        KeyError: 环境变量 MODELSCOPE_API_KEY 未设置。
        requests.HTTPError: 提交任务或查询状态时 HTTP 请求失败。
        RuntimeError: ModelScope 任务执行失败(状态为 FAILED)。
    """
    # 缺失时 KeyError 自动抛出,无需手动处理
    api_key = os.environ["MODELSCOPE_API_KEY"]

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    # 1. 提交异步生成任务
    submit_resp = requests.post(
        f"{BASE_URL}v1/images/generations",
        headers={**headers, "X-ModelScope-Async-Mode": "true"},
        json={"model": model, "prompt": prompt},
        timeout=REQUEST_TIMEOUT,
    )
    submit_resp.raise_for_status()
    task_id = submit_resp.json()["task_id"]

    # 2. 轮询任务状态直到 SUCCEED 或 FAILED
    while True:
        query_resp = requests.get(
            f"{BASE_URL}v1/tasks/{task_id}",
            headers={**headers, "X-ModelScope-Task-Type": "image_generation"},
            timeout=REQUEST_TIMEOUT,
        )
        query_resp.raise_for_status()
        payload = query_resp.json()
        status = payload["task_status"]

        if status == "SUCCEED":
            image_url = payload["output_images"][0]
            image = Image.open(
                BytesIO(requests.get(image_url, timeout=REQUEST_TIMEOUT).content)
            )
            save_path = Path(save_path).resolve()
            image.save(save_path)
            return save_path

        elif status == "FAILED":
            raise RuntimeError(f"ModelScope 任务 {task_id} 执行失败: {payload}")

        time.sleep(poll_interval)

# generate_image(prompt="a sex girl", save_path='a sex girl.jpg')

In [3]:
import os
from pathlib import Path
from typing import Literal

import yaml
from langchain.tools import tool

EXAMPLE_DIR = Path("/home/cooper/githubProjects/agent-deploy-kit/agents/content_build_agent")

print(EXAMPLE_DIR)


@tool
def web_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news"] = "general",
) -> dict:
    """Search the web for current information.

    Args:
        query: The search query (be specific and detailed)
        max_results: Number of results to return (default: 5)
        topic: "general" for most queries, "news" for current events

    Returns:
        Search results with titles, URLs, and content excerpts.
    """
    try:
        from tavily import TavilyClient

        api_key = os.environ.get("TAVILY_API_KEY")
        if not api_key:
            return {"error": "TAVILY_API_KEY not set"}

        client = TavilyClient(api_key=api_key)
        return client.search(query, max_results=max_results, topic=topic)
    except Exception as e:
        return {"error": f"Search failed: {e}"}


@tool
def generate_cover(prompt: str, slug: str) -> str:
    """Generate a cover image for a blog post.

    Args:
        prompt: Detailed description of the image to generate.
        slug: Blog post slug. Image saves to blogs/<slug>/hero.png
    """
    try:
        output_path = EXAMPLE_DIR / "blogs" / slug / "hero.png"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        generate_image(prompt=prompt, save_path=output_path)
        return f"Image saved to {output_path}"
        # from google import genai

        # client = genai.Client()
        # response = client.models.generate_content(
        #     model="gemini-2.5-flash-image",
        #     contents=[prompt],
        # )

        # for part in response.parts:
        #     if part.inline_data is not None:
        #         image = part.as_image()
        #         output_path = EXAMPLE_DIR / "blogs" / slug / "hero.png"
        #         output_path.parent.mkdir(parents=True, exist_ok=True)
        #         image.save(str(output_path))
        #         return f"Image saved to {output_path}"

        # return "No image generated"
    except Exception as e:
        return f"Error: {e}"


@tool
def generate_social_image(prompt: str, platform: str, slug: str) -> str:
    """Generate an image for a social media post.

    Args:
        prompt: Detailed description of the image to generate.
        platform: Either "linkedin" or "tweets"
        slug: Post slug. Image saves to <platform>/<slug>/image.png
    """
    try:
        output_path = EXAMPLE_DIR / platform / slug / "image.png"
        output_path.parent.mkdir(parents=True, exist_ok=True)
        generate_image(prompt=prompt, save_path=output_path)
        return f"Image saved to {output_path}"
    
        # from google import genai

        # client = genai.Client()
        # response = client.models.generate_content(
        #     model="gemini-2.5-flash-image",
        #     contents=[prompt],
        # )

        # for part in response.parts:
        #     if part.inline_data is not None:
        #         image = part.as_image()
        #         output_path = EXAMPLE_DIR / platform / slug / "image.png"
        #         output_path.parent.mkdir(parents=True, exist_ok=True)
        #         image.save(str(output_path))
        #         return f"Image saved to {output_path}"

        # return "No image generated"
    except Exception as e:
        return f"Error: {e}"


def load_subagents(config_path: Path) -> list:
    """Load subagent definitions from YAML and wire up tools.

    Unlike `memory` and `skills`, deep agents do not load subagents from files by default.
    This helper externalizes configuration so you can edit YAML without changing Python code.
    """
    available_tools = {
        "web_search": web_search,
    }

    with open(config_path) as f:
        config = yaml.safe_load(f)

    subagents = []
    for name, spec in config.items():
        subagent = {
            "name": name,
            "description": spec["description"],
            "system_prompt": spec["system_prompt"],
        }
        if "model" in spec:
            subagent["model"] = spec["model"]
        if "tools" in spec:
            subagent["tools"] = [available_tools[t] for t in spec["tools"]]
        subagents.append(subagent)

    return subagents

/home/cooper/githubProjects/agent-deploy-kit/agents/content_build_agent


In [4]:
# load_subagents(EXAMPLE_DIR / "docs/subagents.yaml")

In [5]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from utils.langchain_model import get_singleton_client



def create_content_writer():
    """Create a content writer agent configured by filesystem files."""
    return create_deep_agent(
        model=get_singleton_client(llm_provider="longcat"),
        memory=["./docs/AGENTS.md"],
        skills=["./docs/skills/"],
        tools=[generate_cover, generate_social_image],
        subagents=load_subagents(EXAMPLE_DIR / "docs/subagents.yaml"),
        backend=FilesystemBackend(root_dir=EXAMPLE_DIR, virtual_mode=True),
    )

In [6]:
from langchain.messages import HumanMessage

agent = create_content_writer()
for chunk in  agent.stream(
    {"messages": [HumanMessage(content="Write a blog post about how AI agents are transforming software development")]},
    # config={"configurable": {"thread_id": "content-builder-demo"}},
):
    print(chunk)

{'SkillsMiddleware.before_agent': {'skills_metadata': [{'name': 'blog-post', 'description': 'Writes and structures long-form blog posts, creates tutorial outlines, and optimizes content for SEO with cover image generation. Use when the user asks to write a blog post, article, how-to guide, tutorial, technical writeup, thought leadership piece, or long-form content.', 'path': '/docs/skills/blog-post/SKILL.md', 'metadata': {}, 'license': None, 'compatibility': None, 'allowed_tools': []}, {'name': 'social-media', 'description': 'Drafts engaging social media posts, writes hooks, suggests hashtags, creates thread structures, and generates companion images. Use when the user asks to write a LinkedIn post, tweet, Twitter/X thread, social media caption, social post, or repurpose content for social platforms.', 'path': '/docs/skills/social-media/SKILL.md', 'metadata': {}, 'license': None, 'compatibility': None, 'allowed_tools': []}]}}
{'PatchToolCallsMiddleware.before_agent': None}
{'MemoryMidd